In [0]:
dbutils.widgets.text("bronze_layer", "bronze")
bronze = dbutils.widgets.get("bronze_layer")

dbutils.widgets.text("silver_layer", "silver")
silver = dbutils.widgets.get("silver_layer")

dbutils.widgets.text("schema_name", "fhir")
schema = dbutils.widgets.get("schema_name")

dbutils.widgets.text("tables", "Patient,Encounter,Observation,Condition")
tables_str = dbutils.widgets.get("tables")

Tables = [t.strip() for t in tables_str.split(",")]


In [0]:
table = None
for i in Tables:
    if i.lower() == "condition":
        table = i
        break

if table is None:
    dbutils.notebook.exit("Exiting notebook: Table is not condition")

print(table + " table is present ")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


### Assigning the value to the variable and loading the data into df based on if the bronze table exist or not.

In [0]:
bronze_table = f"{bronze}.{schema}.{table}"
silver_table = f"{silver}.{schema}.{table}"

if spark.catalog.tableExists(silver_table):
    df = spark.table(bronze_table).filter(
        col("load_date") >= date_sub(current_date(), 4)
    )
else:
    df = spark.table(bronze_table)

df.select("load_date").show()


### Exploding and flattening the raw_JSON from df created from bronze table.

In [0]:


parsed_df = df.withColumn(
    "json_data",
    from_json(col("raw_json"), "array<struct<resourceType:string,id:string,meta:struct<lastUpdated:string>,type:string,entry:array<struct<fullUrl:string,resource:struct<resourceType:string,id:string,meta:struct<versionId:string,lastUpdated:string>,subject:struct<reference:string>,encounter:struct<reference:string>,code:struct<text:string>>>>>>")
)

exploded = parsed_df.select(
    explode(col("json_data")).alias("bundle"),
    "extraction_timestamp"
)

entries = exploded.select(
    explode(col("bundle.entry")).alias("entry"),
    "extraction_timestamp"
)

In [0]:
silver_condition = entries.select(
    col("entry.resource.id").alias("condition_id"),
    col("entry.resource.subject.reference").alias("patient_ref"),
    col("entry.resource.encounter.reference").alias("encounter_ref"),
    col("entry.resource.code.text").alias("condition_text"),
    col("entry.resource.meta.lastUpdated").alias("last_updated")
)
silver_condition.show(1)

In [0]:
silver_condition = silver_condition \
    .withColumn("patient_id", split(col("patient_ref"), "/")[1]) \
    .withColumn("encounter_id", split(col("encounter_ref"), "/")[1]) \
    .drop("patient_ref","encounter_ref")

silver_condition.orderBy("last_updated").show(1)

### Removing duplicate PK records based on last_updated date. So, only records having latest last_updated date will be considered.

In [0]:
window = Window.partitionBy("condition_id").orderBy(col("last_updated").desc())

silver_condition = silver_condition \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

silver_condition.show(5)

In [0]:
staging_df = silver_condition

staging_df = staging_df \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

### Checking if silver table exist or not if it exsist than the the code will go to merge in case if silver table now exist df will directly created the table and exit the notebook

In [0]:

if not spark.catalog.tableExists(silver_table):
    staging_df.write.format("delta").saveAsTable(silver_table)
    dbutils.notebook.exit("Exiting notebook: As this was the first time load")

In [0]:
staging_df.createOrReplaceTempView("staging_condition")

### SCD 2 Type loading data into silver table. Therefore, 3 things are handle - 
1. if an update is found for a record it will inactive the older record. 
2. if a new record is found it will insert it. 
3. and than it will insert the update record.

In [0]:
query = f"""MERGE INTO {silver_table} AS tgt
USING staging_condition AS src
ON tgt.condition_id = src.condition_id
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.condition_text <> src.condition_text OR
    tgt.patient_id <> src.patient_id OR
    tgt.encounter_id <> src.encounter_id
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    condition_id,
    patient_id,
    encounter_id,
    condition_text,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.condition_id,
    src.patient_id,
    src.encounter_id,
    src.condition_text,
    src.last_updated,
    src.effective_start_date,
    src.effective_end_date,
    src.is_current
)"""

query1 = f"""
INSERT INTO {silver_table}
SELECT 
    src.condition_id,
    src.patient_id,
    src.encounter_id,
    src.condition_text,
    src.last_updated,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_condition src
JOIN {silver_table} tgt
    ON tgt.condition_id = src.condition_id
WHERE 
    tgt.is_current = false   -- was just inactivated
AND (
    tgt.condition_text <> src.condition_text OR
    tgt.patient_id <> src.patient_id OR
    tgt.encounter_id <> src.encounter_id
)
AND NOT EXISTS (
    SELECT 1
    FROM {silver_table} t2
    WHERE t2.condition_id = src.condition_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)
